# SwinJSCC Base — DIV2K Training → Kodak Evaluation
## Kaggle 2× NVIDIA T4 / PyTorch DataParallel

This notebook establishes the **baseline SwinJSCC model** that we will later modify for the semantic-codebook experiment.

It uses the official `semcomm/SwinJSCC` implementation and keeps the mathematical architecture intact.

### Experiment

```text
DIV2K HR images
      ↓
random 256×256 HR crops
      ↓
official SwinJSCC encoder
      ↓
Channel ModNet + Rate ModNet
      ↓
AWGN channel
      ↓
official SwinJSCC decoder
      ↓
reconstructed image
      ↓
Kodak held-out evaluation
```

### Two-stage training

1. `SwinJSCC_w/o_SAandRA`
   - fixed SNR
   - fixed C
   - baseline pretraining

2. `SwinJSCC_w/_SAandRA`
   - Channel ModNet
   - Rate ModNet
   - multiple SNRs
   - multiple C values

### Important implementation decisions

The original official `network.py` returns Python/numeric metrics together with tensors. That output structure is unsuitable for reliable `nn.DataParallel` gathering.

Therefore this notebook calls the **official encoder, channel and decoder directly** and returns tensors only.

The official loss/distortion implementation is also used rather than replacing it with a different MSE definition.

The official encoder contains device-specific `.cuda()` calls in the Rate ModNet. Those are patched to use the input tensor's device. This does not change the model mathematics; it makes the official implementation safe for two-GPU replication.


In [ ]:
import os
import sys
import math
import time
import random
import logging
import shutil
import subprocess
import importlib
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from PIL import Image

import torch
import torch.nn as nn
import torch.nn.functional as F

from torch.utils.data import Dataset, DataLoader
from torchvision import transforms

print("Python :", sys.version)
print("PyTorch:", torch.__version__)
print("CUDA   :", torch.version.cuda)
print("GPU count:", torch.cuda.device_count())

for i in range(torch.cuda.device_count()):
    print(f"GPU {i}: {torch.cuda.get_device_name(i)}")


## 1. Dataset paths

These are the exact paths requested for this experiment.

DIV2K is used for training.

Kodak is **not** used for training. It remains the held-out evaluation dataset.


In [ ]:
KAGGLE_KODAK_DIR = Path(
    "/kaggle/input/datasets/sherylmehta/kodak-dataset"
)

DIV2K_DIR = Path(
    "/kaggle/input/notebooks/jagan028/div2k-dataset-generation-for-isr"
)

IMAGE_SIZE = 256

if not DIV2K_DIR.exists():
    raise FileNotFoundError(
        f"DIV2K dataset not found: {DIV2K_DIR}"
    )

if not KAGGLE_KODAK_DIR.exists():
    raise FileNotFoundError(
        f"Kodak dataset not found: {KAGGLE_KODAK_DIR}"
    )

print("DIV2K :", DIV2K_DIR)
print("Kodak :", KAGGLE_KODAK_DIR)


In [ ]:
IMAGE_EXTENSIONS = {
    ".png", ".jpg", ".jpeg", ".bmp",
    ".webp", ".tif", ".tiff"
}

def discover_images(root):
    return sorted([
        p for p in root.rglob("*")
        if p.is_file()
        and p.suffix.lower() in IMAGE_EXTENSIONS
    ])

div2k_files_all = discover_images(DIV2K_DIR)
kodak_files_all = discover_images(KAGGLE_KODAK_DIR)

print("All files found under DIV2K path:", len(div2k_files_all))
print("All files found under Kodak path:", len(kodak_files_all))

print("\nDIV2K examples:")
for p in div2k_files_all[:15]:
    print(" ", p)

print("\nKodak examples:")
for p in kodak_files_all[:15]:
    print(" ", p)


In [ ]:
def image_metadata(files, max_files=None):
    records = []

    selected = files if max_files is None else files[:max_files]

    for p in selected:
        try:
            with Image.open(p) as im:
                records.append({
                    "path": str(p),
                    "width": im.width,
                    "height": im.height,
                    "format": im.format,
                })
        except Exception as exc:
            print("Unreadable:", p, "|", exc)

    return pd.DataFrame(records)

div2k_meta = image_metadata(div2k_files_all)
kodak_meta = image_metadata(kodak_files_all)

print("DIV2K readable images:", len(div2k_meta))
print("Kodak readable images:", len(kodak_meta))

if len(div2k_meta):
    print("\nDIV2K dimensions:")
    print(
        div2k_meta[
            ["width", "height"]
        ].describe()
    )

if len(kodak_meta):
    print("\nKodak dimensions:")
    print(
        kodak_meta[
            ["width", "height"]
        ].describe()
    )


In [ ]:
# Select only HR images that can provide a 256x256 crop.
# This prevents generated low-resolution ISR products from
# accidentally becoming SwinJSCC training data.

def is_valid_hr(path, minimum=256):
    try:
        with Image.open(path) as im:
            return (
                im.width >= minimum
                and im.height >= minimum
            )
    except Exception:
        return False

train_files = [
    p for p in div2k_files_all
    if is_valid_hr(p, IMAGE_SIZE)
]

if not train_files:
    raise RuntimeError(
        "No usable DIV2K HR images >= 256x256 were found."
    )

test_files = [
    p for p in kodak_files_all
    if is_valid_hr(p, 128)
]

print("DIV2K HR training images:", len(train_files))
print("Kodak evaluation images:", len(test_files))

print("\nTraining image resolution examples:")
for p in train_files[:10]:
    with Image.open(p) as im:
        print(f"  {p.name}: {im.size}")


## 2. Data pipeline

For training, each HR DIV2K image is randomly cropped to 256×256.

This means one 2K image can generate many different training samples over different epochs without physically creating a giant patch dataset.

Kodak is kept as a whole-image evaluation set, with dimensions cropped to multiples of 128 as in the official evaluation loader.


In [ ]:
class DIV2KRandomCrop(Dataset):

    def __init__(self, files, crop_size=256):
        self.files = list(files)
        self.crop_size = crop_size
        self.to_tensor = transforms.ToTensor()

    def __len__(self):
        return len(self.files)

    def __getitem__(self, index):
        path = self.files[index]

        with Image.open(path) as im:
            im = im.convert("RGB")

            w, h = im.size

            if w < self.crop_size or h < self.crop_size:
                raise RuntimeError(
                    f"Image smaller than crop size: "
                    f"{path} -> {im.size}"
                )

            x = random.randint(
                0,
                w - self.crop_size
            )

            y = random.randint(
                0,
                h - self.crop_size
            )

            crop = im.crop((
                x,
                y,
                x + self.crop_size,
                y + self.crop_size
            ))

            if random.random() < 0.5:
                crop = crop.transpose(
                    Image.Transpose.FLIP_LEFT_RIGHT
                )

            if random.random() < 0.5:
                crop = crop.transpose(
                    Image.Transpose.FLIP_TOP_BOTTOM
                )

            return self.to_tensor(crop)


class KodakEvaluation(Dataset):

    def __init__(self, files):
        self.files = list(files)

    def __len__(self):
        return len(self.files)

    def __getitem__(self, index):
        path = self.files[index]

        with Image.open(path) as im:
            im = im.convert("RGB")

            w, h = im.size

            w = w - (w % 128)
            h = h - (h % 128)

            if w < 128 or h < 128:
                raise RuntimeError(
                    f"Kodak image is too small: "
                    f"{path} -> {im.size}"
                )

            left = (im.width - w) // 2
            top = (im.height - h) // 2

            im = im.crop((
                left,
                top,
                left + w,
                top + h
            ))

            tensor = transforms.ToTensor()(im)

        return tensor, path.name


train_dataset = DIV2KRandomCrop(
    train_files,
    IMAGE_SIZE
)

test_dataset = KodakEvaluation(
    test_files
)

BATCH_SIZE = 8
NUM_WORKERS = min(4, os.cpu_count() or 1)

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    drop_last=True,
    num_workers=NUM_WORKERS,
    pin_memory=True,
    persistent_workers=NUM_WORKERS > 0,
)

test_loader = DataLoader(
    test_dataset,
    batch_size=1,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=True,
    persistent_workers=NUM_WORKERS > 0,
)

print("Train dataset length:", len(train_dataset))
print("Test dataset length :", len(test_dataset))
print("Batch size          :", BATCH_SIZE)
print("Workers             :", NUM_WORKERS)


## 3. Clone and inspect the official SwinJSCC source

We intentionally inspect the source before constructing the model.

The files that matter are:

```text
net/encoder.py
net/decoder.py
net/channel.py
net/network.py
loss/distortion.py
data/datasets.py
```

The architecture itself is not rewritten here.


In [ ]:
REPO_DIR = Path(
    "/kaggle/working/SwinJSCC_official"
)

if REPO_DIR.exists():
    shutil.rmtree(REPO_DIR)

result = subprocess.run(
    [
        "git",
        "clone",
        "--depth",
        "1",
        "https://github.com/semcomm/SwinJSCC.git",
        str(REPO_DIR),
    ],
    capture_output=True,
    text=True,
)

if result.returncode != 0:
    print(result.stdout)
    print(result.stderr)
    raise RuntimeError(
        "Could not clone official SwinJSCC repository."
    )

if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))

print("Repository:", REPO_DIR)


In [ ]:
# Inspect the relevant official files before importing them.

important_files = [
    "net/encoder.py",
    "net/decoder.py",
    "net/channel.py",
    "net/network.py",
    "loss/distortion.py",
    "data/datasets.py",
]

for relative in important_files:
    path = REPO_DIR / relative
    print(
        f"{relative:<25} exists={path.exists()} "
        f"size={path.stat().st_size if path.exists() else 'N/A'}"
    )


In [ ]:
from net.encoder import create_encoder
from net.decoder import create_decoder
from net.channel import Channel

print("Official SwinJSCC imports successful.")


## 4. Make the official Rate ModNet device-safe

The official Rate ModNet contains `.cuda()` calls when constructing channel-index/mask tensors.

That is harmless on a single GPU but is not correct for a replicated module if a replica is executing on GPU 1.

We make the minimum device-only patch:

```text
.cuda()
   ↓
device = x.device
   ↓
tensor(..., device=device)
```

No layer, parameter, activation function, sorting rule, mask logic, or rate-selection rule is changed.

The patched source is loaded from a separate Kaggle working copy so the original repository remains available for comparison.


In [ ]:
# Show every .cuda() occurrence in the official encoder.
encoder_source_path = REPO_DIR / "net" / "encoder.py"
encoder_source = encoder_source_path.read_text(
    encoding="utf-8"
)

for line_no, line in enumerate(
    encoder_source.splitlines(),
    start=1
):
    if ".cuda()" in line:
        print(
            f"{line_no:4d}: {line}"
        )


In [ ]:
# Patch only device-specific tensor creation in the official
# Rate ModNet. We make a backup first.

encoder_backup = encoder_source_path.with_suffix(
    ".py.original"
)

encoder_backup.write_text(
    encoder_source,
    encoding="utf-8"
)

patched = encoder_source

# Official code creates a c_indices tensor with .cuda().
patched = re.sub(
    r"torch\.zeros\(.*?\)\.reshape\(-1\)\.cuda\(\)",
    lambda m: m.group(0).replace(
        ".cuda()",
        ".to(x.device)"
    ),
    patched,
)

# More robust direct replacements for common official forms.
patched = patched.replace(
    "torch.zeros(mask.size()).reshape(-1).cuda()",
    "torch.zeros(mask.size(), device=x.device).reshape(-1)"
)

patched = patched.replace(
    "torch.arange(0, c).cuda()",
    "torch.arange(0, c, device=x.device)"
)

patched = patched.replace(
    "torch.arange(0, c).int().cuda()",
    "torch.arange(0, c, device=x.device).int()"
)

# If the exact repository version contains c_indices.cuda(),
# use the surrounding tensor's device.
patched = patched.replace(
    "c_indices = c_indices + add.int().cuda()",
    "c_indices = c_indices + add.int().to(x.device)"
)

encoder_source_path.write_text(
    patched,
    encoding="utf-8"
)

remaining_cuda = [
    (i, line)
    for i, line in enumerate(
        patched.splitlines(),
        start=1
    )
    if ".cuda()" in line
]

print(
    "Remaining .cuda() occurrences in encoder.py:",
    len(remaining_cuda)
)

for item in remaining_cuda:
    print(item)


In [ ]:
# Reload the modified encoder module.
import net.encoder as encoder_module

encoder_module = importlib.reload(
    encoder_module
)

create_encoder = encoder_module.create_encoder

print("Patched official encoder reloaded.")


## 5. Exact HR Base configuration

We use the full Base architecture:

```text
Encoder:
  dim 3    @ 256×256, depth 2
  dim 128  @ 128×128, depth 2
  dim 192  @ 64×64,   depth 6
  dim 256  @ 32×32,   depth 2

Decoder:
  dim 320  @ 16×16,   depth 2
  dim 256  @ 32×32,   depth 6
  dim 192  @ 64×64,   depth 2
  dim 128  @ 128×128, depth 2

Final latent:
  16×16×320 = 81,920 continuous values
```

The SA+RA model uses `C=None` at construction because Rate ModNet determines the active channel count.


In [ ]:
MODEL_BASE = "SwinJSCC_w/o_SAandRA"
MODEL_FULL = "SwinJSCC_w/_SAandRA"

EMBED_DIMS = [128, 192, 256, 320]

ENCODER_DEPTHS = [2, 2, 6, 2]
DECODER_DEPTHS = [2, 6, 2, 2]

NUM_HEADS = [4, 6, 8, 10]

PATCH_SIZE = 2
WINDOW_SIZE = 8
MLP_RATIO = 4.0

DOWNSAMPLE = 4

PRETRAIN_SNR = 10
PRETRAIN_C = 96

TRAIN_SNRS = [1, 4, 7, 10, 13]
TRAIN_RATES = [32, 64, 96, 128, 192]

print("MODEL_BASE:", MODEL_BASE)
print("MODEL_FULL:", MODEL_FULL)
print("Encoder depths:", ENCODER_DEPTHS)
print("Decoder depths:", DECODER_DEPTHS)
print("Embedding dimensions:", EMBED_DIMS)
print("Heads:", NUM_HEADS)


In [ ]:
def encoder_kwargs(model_name, C):
    return dict(
        model=model_name,
        img_size=(IMAGE_SIZE, IMAGE_SIZE),
        patch_size=PATCH_SIZE,
        in_chans=3,
        embed_dims=EMBED_DIMS,
        depths=ENCODER_DEPTHS,
        num_heads=NUM_HEADS,
        C=C,
        window_size=WINDOW_SIZE,
        mlp_ratio=MLP_RATIO,
        qkv_bias=True,
        qk_scale=None,
        norm_layer=nn.LayerNorm,
        patch_norm=True,
    )


def decoder_kwargs(model_name, C):
    return dict(
        model=model_name,
        img_size=(IMAGE_SIZE, IMAGE_SIZE),
        embed_dims=EMBED_DIMS[::-1],
        depths=DECODER_DEPTHS,
        num_heads=NUM_HEADS[::-1],
        C=C,
        window_size=WINDOW_SIZE,
        mlp_ratio=MLP_RATIO,
        qkv_bias=True,
        qk_scale=None,
        norm_layer=nn.LayerNorm,
        patch_norm=True,
    )


ENC_BASE = encoder_kwargs(
    MODEL_BASE,
    PRETRAIN_C
)

DEC_BASE = decoder_kwargs(
    MODEL_BASE,
    PRETRAIN_C
)

ENC_FULL = encoder_kwargs(
    MODEL_FULL,
    None
)

DEC_FULL = decoder_kwargs(
    MODEL_FULL,
    None
)


## 6. Official Channel configuration

The official `Channel` constructor expects fields including:

```text
args.channel_type
args.multiple_snr
config.device
config.logger
config.pass_channel
```

We provide those explicitly.

For AWGN, the official channel's noise generation follows the incoming feature tensor's device, which is important for DataParallel.


In [ ]:
class ChannelArgs:
    pass


class ChannelConfig:
    pass


def make_channel_args(
    model_name,
    snr_values,
    rate_values
):
    args = ChannelArgs()

    args.model = model_name
    args.channel_type = "awgn"

    args.multiple_snr = ",".join(
        map(str, snr_values)
    )

    args.C = ",".join(
        map(str, rate_values)
    )

    return args


def make_channel_config():

    config = ChannelConfig()

    config.pass_channel = True

    config.device = torch.device(
        "cuda:0"
        if torch.cuda.is_available()
        else "cpu"
    )

    config.logger = None

    return config


## 7. DataParallel-safe wrapper around the official modules

This wrapper does **not** reimplement SwinJSCC.

It composes:

```text
official encoder
official Channel
official decoder
```

and returns only:

```text
reconstruction
feature
mask
```

all tensors.

This avoids the original `network.py` DataParallel gathering problem caused by Python scalar metrics.

The SA+RA path is:

```text
image
  ↓
encoder
  ↓
feature + mask
  ↓
Channel
  ↓
decoder
  ↓
reconstruction
```


In [ ]:
class OfficialSwinJSCCWrapper(nn.Module):

    def __init__(
        self,
        model_name,
        enc_kwargs,
        dec_kwargs,
        snrs,
        rates,
    ):
        super().__init__()

        self.model_name = model_name

        self.encoder = create_encoder(
            **enc_kwargs
        )

        self.decoder = create_decoder(
            **dec_kwargs
        )

        self.channel_args = make_channel_args(
            model_name,
            snrs,
            rates
        )

        self.channel_config = (
            make_channel_config()
        )

        self.channel = Channel(
            self.channel_args,
            self.channel_config
        )

        self.H = 0
        self.W = 0

    def update_resolution(self, H, W):

        if (
            self.H != H
            or self.W != W
        ):
            self.encoder.update_resolution(
                H,
                W
            )

            self.decoder.update_resolution(
                H // (2 ** DOWNSAMPLE),
                W // (2 ** DOWNSAMPLE)
            )

            self.H = H
            self.W = W

    def forward(
        self,
        input_image,
        snr,
        rate
    ):

        H = input_image.shape[-2]
        W = input_image.shape[-1]

        self.update_resolution(H, W)

        if self.model_name in [
            "SwinJSCC_w/o_SAandRA",
            "SwinJSCC_w/_SA",
        ]:

            feature = self.encoder(
                input_image,
                snr,
                rate,
                self.model_name
            )

            noisy_feature = (
                self.channel.forward(
                    feature,
                    snr
                )
            )

            mask = torch.ones_like(
                feature
            )

        elif self.model_name in [
            "SwinJSCC_w/_RA",
            "SwinJSCC_w/_SAandRA",
        ]:

            feature, mask = (
                self.encoder(
                    input_image,
                    snr,
                    rate,
                    self.model_name
                )
            )

            avg_pwr = (
                torch.sum(feature ** 2)
                /
                mask.sum().clamp_min(
                    1e-12
                )
            )

            noisy_feature = (
                self.channel.forward(
                    feature,
                    snr,
                    avg_pwr
                )
            )

            noisy_feature = (
                noisy_feature * mask
            )

        else:
            raise ValueError(
                f"Unsupported model: "
                f"{self.model_name}"
            )

        reconstruction = (
            self.decoder(
                noisy_feature,
                snr,
                self.model_name
            )
        )

        reconstruction = (
            reconstruction.clamp(
                0.0,
                1.0
            )
        )

        return (
            reconstruction,
            feature,
            mask
        )


In [ ]:
def make_dataparallel(model):

    if not torch.cuda.is_available():
        return model

    model = model.cuda(0)

    if torch.cuda.device_count() >= 2:

        print(
            "Wrapping with DataParallel:",
            list(
                range(
                    torch.cuda.device_count()
                )
            )
        )

        model = nn.DataParallel(
            model,
            device_ids=list(
                range(
                    torch.cuda.device_count()
                )
            ),
            output_device=0
        )

    else:
        print("Only one GPU available.")

    return model


baseline_model = make_dataparallel(
    OfficialSwinJSCCWrapper(
        MODEL_BASE,
        ENC_BASE,
        DEC_BASE,
        [PRETRAIN_SNR],
        [PRETRAIN_C]
    )
)

sara_model = make_dataparallel(
    OfficialSwinJSCCWrapper(
        MODEL_FULL,
        ENC_FULL,
        DEC_FULL,
        TRAIN_SNRS,
        TRAIN_RATES
    )
)

print(
    "baseline DataParallel:",
    isinstance(
        baseline_model,
        nn.DataParallel
    )
)

print(
    "SA+RA DataParallel:",
    isinstance(
        sara_model,
        nn.DataParallel
    )
)


In [ ]:
# ============================================================
# CRITICAL 2-GPU FORWARD TEST
# ============================================================

if torch.cuda.device_count() >= 2:
    smoke_batch_size = 8
else:
    smoke_batch_size = 2

smoke_images = next(
    iter(train_loader)
)[:smoke_batch_size].cuda(
    0,
    non_blocking=True
)

baseline_model.eval()
sara_model.eval()

with torch.no_grad():

    baseline_recon, baseline_feature, baseline_mask = (
        baseline_model(
            smoke_images,
            PRETRAIN_SNR,
            PRETRAIN_C
        )
    )

    sara_recon, sara_feature, sara_mask = (
        sara_model(
            smoke_images,
            10,
            96
        )
    )

print("Input:", tuple(smoke_images.shape))

print(
    "Baseline recon:",
    tuple(baseline_recon.shape)
)

print(
    "Baseline feature:",
    tuple(baseline_feature.shape)
)

print(
    "SA+RA recon:",
    tuple(sara_recon.shape)
)

print(
    "SA+RA feature:",
    tuple(sara_feature.shape)
)

print(
    "SA+RA mask:",
    tuple(sara_mask.shape)
)

assert baseline_recon.shape == smoke_images.shape
assert sara_recon.shape == smoke_images.shape
assert sara_feature.ndim == 3
assert sara_mask.shape == sara_feature.shape
assert sara_feature.shape[-1] == 320

print("\nFORWARD TEST PASSED.")


In [ ]:
# GPU memory check after the two-GPU forward.

for i in range(torch.cuda.device_count()):

    print(
        f"GPU {i}: "
        f"allocated={torch.cuda.memory_allocated(i)/1024**3:.2f} GB | "
        f"reserved={torch.cuda.memory_reserved(i)/1024**3:.2f} GB"
    )

print(
    "\nDuring actual training, use !nvidia-smi "
    "to observe both GPUs."
)


# 8. Official distortion loss

The official implementation uses its `Distortion` class rather than simply calling an arbitrary PyTorch MSE function.

We import that class and reproduce its loss call inside the training loop.

This matters because our goal at this stage is a trustworthy SwinJSCC baseline. We should not change the training objective before introducing our semantic-codebook modification.


In [ ]:
from loss.distortion import Distortion

print("Official Distortion imported:", Distortion)


In [ ]:
# Construct the same style of args object needed by the
# official distortion implementation.

class LossArgs:
    pass


loss_args = LossArgs()

# The official distortion class checks this option.
loss_args.distortion_metric = "MSE"

distortion_loss = Distortion(
    loss_args
)

print("Official distortion loss constructed.")


# 9. Stage 1 — `SwinJSCC_w/o_SAandRA`

Fixed:

```text
SNR = 10 dB
C   = 96
```

This is the baseline pretraining stage.


In [ ]:
PRETRAIN_EPOCHS = 10
LR_PRETRAIN = 1e-4

optimizer_base = torch.optim.Adam(
    baseline_model.parameters(),
    lr=LR_PRETRAIN
)

use_amp = torch.cuda.is_available()

scaler_base = torch.amp.GradScaler(
    "cuda",
    enabled=use_amp
)

pretrain_history = []

for epoch in range(PRETRAIN_EPOCHS):

    baseline_model.train()

    running_loss = 0.0
    seen = 0

    start = time.time()

    for images in train_loader:

        images = images.cuda(
            0,
            non_blocking=True
        )

        optimizer_base.zero_grad(
            set_to_none=True
        )

        with torch.amp.autocast(
            "cuda",
            enabled=use_amp
        ):

            reconstruction, _, _ = (
                baseline_model(
                    images,
                    PRETRAIN_SNR,
                    PRETRAIN_C
                )
            )

            loss = distortion_loss(
                images,
                reconstruction.clamp(
                    0.0,
                    1.0
                )
            )

        scaler_base.scale(
            loss
        ).backward()

        scaler_base.unscale_(
            optimizer_base
        )

        torch.nn.utils.clip_grad_norm_(
            baseline_model.parameters(),
            1.0
        )

        scaler_base.step(
            optimizer_base
        )

        scaler_base.update()

        running_loss += (
            float(loss.detach())
            * images.size(0)
        )

        seen += images.size(0)

    epoch_loss = (
        running_loss
        /
        max(seen, 1)
    )

    pretrain_history.append(
        epoch_loss
    )

    print(
        f"[BASE] "
        f"Epoch {epoch+1}/{PRETRAIN_EPOCHS} | "
        f"Loss={epoch_loss:.6f} | "
        f"{time.time()-start:.1f}s"
    )


In [ ]:
plt.figure(figsize=(8, 5))

plt.plot(
    range(
        1,
        len(pretrain_history) + 1
    ),
    pretrain_history,
    marker="o"
)

plt.xlabel("Epoch")
plt.ylabel("Official distortion loss")
plt.title("SwinJSCC Baseline Pretraining")
plt.grid(True)
plt.show()


# 10. Stage 2 — `SwinJSCC_w/_SAandRA`

Now initialize the full adaptive model with compatible weights from Stage 1.

The training conditions are:

```text
SNR = 1, 4, 7, 10, 13 dB

C = 32, 64, 96, 128, 192
```

The Rate ModNet controls which latent channels are active.


In [ ]:
def unwrap(model):
    return (
        model.module
        if isinstance(
            model,
            nn.DataParallel
        )
        else model
    )


base_core = unwrap(
    baseline_model
)

sara_core = unwrap(
    sara_model
)

base_state = base_core.state_dict()
sara_state = sara_core.state_dict()

compatible = {
    key: value
    for key, value in base_state.items()
    if (
        key in sara_state
        and sara_state[key].shape == value.shape
    )
}

sara_state.update(
    compatible
)

sara_core.load_state_dict(
    sara_state,
    strict=False
)

print(
    "Compatible tensors transferred:",
    len(compatible)
)


In [ ]:
SA_RA_EPOCHS = 10
LR_SA_RA = 1e-4

optimizer_sara = torch.optim.Adam(
    sara_model.parameters(),
    lr=LR_SA_RA
)

scaler_sara = torch.amp.GradScaler(
    "cuda",
    enabled=use_amp
)

sara_history = []

for epoch in range(SA_RA_EPOCHS):

    sara_model.train()

    running_loss = 0.0
    seen = 0

    start = time.time()

    for images in train_loader:

        images = images.cuda(
            0,
            non_blocking=True
        )

        snr = random.choice(
            TRAIN_SNRS
        )

        rate = random.choice(
            TRAIN_RATES
        )

        optimizer_sara.zero_grad(
            set_to_none=True
        )

        with torch.amp.autocast(
            "cuda",
            enabled=use_amp
        ):

            reconstruction, _, _ = (
                sara_model(
                    images,
                    snr,
                    rate
                )
            )

            loss = distortion_loss(
                images,
                reconstruction.clamp(
                    0.0,
                    1.0
                )
            )

        scaler_sara.scale(
            loss
        ).backward()

        scaler_sara.unscale_(
            optimizer_sara
        )

        torch.nn.utils.clip_grad_norm_(
            sara_model.parameters(),
            1.0
        )

        scaler_sara.step(
            optimizer_sara
        )

        scaler_sara.update()

        running_loss += (
            float(loss.detach())
            * images.size(0)
        )

        seen += images.size(0)

    epoch_loss = (
        running_loss
        /
        max(seen, 1)
    )

    sara_history.append(
        epoch_loss
    )

    print(
        f"[SA+RA] "
        f"Epoch {epoch+1}/{SA_RA_EPOCHS} | "
        f"Loss={epoch_loss:.6f} | "
        f"{time.time()-start:.1f}s"
    )


In [ ]:
plt.figure(figsize=(8, 5))

plt.plot(
    range(
        1,
        len(sara_history) + 1
    ),
    sara_history,
    marker="o"
)

plt.xlabel("Epoch")
plt.ylabel("Official distortion loss")
plt.title("SwinJSCC SA+RA Training")
plt.grid(True)
plt.show()


# 11. Kodak evaluation

We now evaluate the full model on Kodak.

For each combination:

\[
SNR\in\{1,4,7,10,13\}
\]

and

\[
C\in\{32,64,96,128,192\}.
\]

We report MSE and PSNR.


In [ ]:
@torch.no_grad()
def evaluate_model(
    model,
    loader,
    snr,
    rate
):

    model.eval()

    total_mse_numerator = 0.0
    total_elements = 0

    for images, names in loader:

        images = images.cuda(
            0,
            non_blocking=True
        )

        reconstruction, _, _ = (
            model(
                images,
                snr,
                rate
            )
        )

        total_mse_numerator += (
            F.mse_loss(
                reconstruction,
                images,
                reduction="sum"
            ).item()
        )

        total_elements += images.numel()

    mse_value = (
        total_mse_numerator
        /
        max(total_elements, 1)
    )

    psnr_value = (
        10
        * math.log10(
            1.0
            /
            max(mse_value, 1e-12)
        )
    )

    return mse_value, psnr_value


evaluation_rows = []

for snr in TRAIN_SNRS:

    for rate in TRAIN_RATES:

        mse_value, psnr_value = (
            evaluate_model(
                sara_model,
                test_loader,
                snr,
                rate
            )
        )

        cbr = (
            rate
            /
            (
                2
                * 3
                * 2 ** (
                    2 * DOWNSAMPLE
                )
            )
        )

        evaluation_rows.append({
            "SNR_dB": snr,
            "C": rate,
            "CBR": cbr,
            "MSE": mse_value,
            "PSNR_dB": psnr_value,
        })

        print(
            f"SNR={snr:>2} dB | "
            f"C={rate:>3} | "
            f"CBR={cbr:.6f} | "
            f"PSNR={psnr_value:.3f} dB"
        )

evaluation_df = pd.DataFrame(
    evaluation_rows
)

display(evaluation_df)


In [ ]:
heatmap = evaluation_df.pivot(
    index="SNR_dB",
    columns="C",
    values="PSNR_dB"
)

plt.figure(figsize=(9, 5))

plt.imshow(
    heatmap.values,
    aspect="auto"
)

plt.xticks(
    range(len(heatmap.columns)),
    heatmap.columns
)

plt.yticks(
    range(len(heatmap.index)),
    heatmap.index
)

plt.xlabel("C")
plt.ylabel("SNR (dB)")
plt.title("SwinJSCC SA+RA — Kodak PSNR")

plt.colorbar(
    label="PSNR (dB)"
)

plt.show()


In [ ]:
plt.figure(figsize=(9, 6))

for snr in TRAIN_SNRS:

    subset = (
        evaluation_df[
            evaluation_df["SNR_dB"] == snr
        ]
        .sort_values("CBR")
    )

    plt.plot(
        subset["CBR"],
        subset["PSNR_dB"],
        marker="o",
        label=f"SNR={snr} dB"
    )

plt.xlabel("CBR")
plt.ylabel("PSNR (dB)")
plt.title("SwinJSCC Rate–Distortion Curve")
plt.grid(True)
plt.legend()
plt.show()


# 12. Latent and communication-rate accounting

For a 256×256 RGB image:

\[
256	imes256	imes3	imes8
=
1,572,864	ext{ bits}
=
192	ext{ KiB}.
\]

The final continuous Swin latent is:

\[
16	imes16	imes320
=
81,920
\]

FP32 values.

Therefore the raw FP32 tensor occupies:

\[
81,920	imes32
=
2,621,440	ext{ bits}
=
320	ext{ KiB}.
\]

This **does not mean SwinJSCC has a negative compression ratio**.

That 320 KiB number is simply the size of the latent if we store the continuous PyTorch representation as ordinary FP32 memory.

SwinJSCC instead models transmission using its communication rate:

\[
CBR =
rac{C}
{2\cdot3\cdot2^{2i}}
\]

with \(i=4\) for the HR configuration:

\[
oxed{CBR=C/1536}.
\]

For the adaptive model, Rate ModNet selects the active \(C\) channels.


In [ ]:
# ============================================================
# REPRESENTATION REPORT
# ============================================================

# Use one 256x256 crop so the spatial latent size is known exactly.

sample_image = next(
    iter(train_loader)
)[0:1].cuda(0)

with torch.no_grad():

    reconstruction, feature, mask = (
        sara_model(
            sample_image,
            10,
            96
        )
    )

B, N, D = feature.shape

original_bits = (
    IMAGE_SIZE
    * IMAGE_SIZE
    * 3
    * 8
)

continuous_latent_values = (
    N * D
)

continuous_fp32_bits = (
    continuous_latent_values
    * 32
)

active_values = int(
    round(
        float(
            mask[0].sum().item()
        )
    )
)

active_fp32_bits = (
    active_values
    * 32
)

requested_C = 96

official_cbr = (
    requested_C
    /
    (
        2
        * 3
        * 2 ** (
            2 * DOWNSAMPLE
        )
    )
)

print("=" * 90)
print("SWINJSCC REPRESENTATION REPORT")
print("=" * 90)

print(
    "Original image:               "
    f"{original_bits:,} bits "
    f"({original_bits/8/1024:.2f} KiB)"
)

print(
    "Continuous latent shape:      "
    f"{tuple(feature.shape)}"
)

print(
    "Continuous latent values:     "
    f"{continuous_latent_values:,}"
)

print(
    "Continuous FP32 latent:       "
    f"{continuous_fp32_bits:,} bits "
    f"({continuous_fp32_bits/8/1024:.2f} KiB)"
)

print(
    "Requested C:                  "
    f"{requested_C}"
)

print(
    "Active latent values:         "
    f"{active_values:,}"
)

print(
    "Active FP32 representation:   "
    f"{active_fp32_bits:,} bits "
    f"({active_fp32_bits/8/1024:.2f} KiB)"
)

print(
    "Official SwinJSCC CBR:        "
    f"{official_cbr:.6f}"
)

print(
    "CBR percentage:               "
    f"{official_cbr*100:.3f}%"
)

print("=" * 90)

print(
    "\nNOTE:"
)

print(
    "Continuous FP32 size is a memory representation."
)

print(
    "Official CBR is the communication-rate measure."
)

print(
    "They must not be treated as the same quantity."
)


In [ ]:
rate_rows = []

for C in TRAIN_RATES:

    channel_symbols = (
        16
        * 16
        * C
    )

    cbr = (
        C
        /
        (
            2
            * 3
            * 2 ** (
                2 * DOWNSAMPLE
            )
        )
    )

    active_fp32_bits = (
        channel_symbols
        * 32
    )

    rate_rows.append({
        "C": C,
        "CBR": cbr,
        "CBR_percent": 100 * cbr,
        "channel_symbols_per_image": channel_symbols,
        "active_FP32_KiB": (
            active_fp32_bits
            / 8
            / 1024
        ),
    })

rate_df = pd.DataFrame(rate_rows)

display(rate_df)


# 13. Reconstruction visualization


In [ ]:
@torch.no_grad()
def reconstruct_one(
    model,
    image,
    snr,
    rate
):

    model.eval()

    output, _, _ = (
        model(
            image.unsqueeze(0).cuda(0),
            snr,
            rate
        )
    )

    return output[0].cpu()


count = min(
    4,
    len(test_dataset)
)

fig, axes = plt.subplots(
    2,
    count,
    figsize=(4 * count, 7)
)

if count == 1:
    axes = np.asarray(
        axes
    ).reshape(2, 1)

for i in range(count):

    original, name = (
        test_dataset[i]
    )

    reconstruction = (
        reconstruct_one(
            sara_model,
            original,
            10,
            96
        )
    )

    axes[0, i].imshow(
        original.permute(
            1, 2, 0
        )
    )

    axes[0, i].set_title(
        f"Original\n{name}"
    )

    axes[0, i].axis("off")

    axes[1, i].imshow(
        reconstruction.permute(
            1, 2, 0
        )
    )

    axes[1, i].set_title(
        "SwinJSCC\nSNR=10 dB, C=96"
    )

    axes[1, i].axis("off")

plt.tight_layout()
plt.show()


# 14. Save experiment artifacts

The checkpoint contains the architecture configuration and model state.

The CSV contains the complete Kodak evaluation grid.

The compression CSV contains the CBR/latent accounting.


In [ ]:
EXPORT_DIR = Path(
    "/kaggle/working/swinjscc_final_results"
)

EXPORT_DIR.mkdir(
    parents=True,
    exist_ok=True
)

base_core = unwrap(
    baseline_model
)

sara_core = unwrap(
    sara_model
)

torch.save(
    {
        "model": MODEL_BASE,
        "state_dict": base_core.state_dict(),
        "image_size": IMAGE_SIZE,
        "embed_dims": EMBED_DIMS,
        "encoder_depths": ENCODER_DEPTHS,
        "decoder_depths": DECODER_DEPTHS,
        "num_heads": NUM_HEADS,
        "snr": PRETRAIN_SNR,
        "C": PRETRAIN_C,
        "training_dataset": str(DIV2K_DIR),
    },
    EXPORT_DIR
    / "SwinJSCC_baseline_DIV2K.pt"
)

torch.save(
    {
        "model": MODEL_FULL,
        "state_dict": sara_core.state_dict(),
        "image_size": IMAGE_SIZE,
        "embed_dims": EMBED_DIMS,
        "encoder_depths": ENCODER_DEPTHS,
        "decoder_depths": DECODER_DEPTHS,
        "num_heads": NUM_HEADS,
        "snrs": TRAIN_SNRS,
        "rates": TRAIN_RATES,
        "training_dataset": str(DIV2K_DIR),
        "evaluation_dataset": str(KAGGLE_KODAK_DIR),
    },
    EXPORT_DIR
    / "SwinJSCC_SA_RA_DIV2K_Kodak.pt"
)

evaluation_df.to_csv(
    EXPORT_DIR
    / "kodak_evaluation.csv",
    index=False
)

rate_df.to_csv(
    EXPORT_DIR
    / "rate_and_latent_table.csv",
    index=False
)

print("Saved artifacts:")
for p in EXPORT_DIR.iterdir():
    print(" ", p)


# 15. Final sanity summary

At this point we have a baseline that can be used for the next stage of the project.

```text
DIV2K
  ↓
SwinJSCC Base
  ↓
continuous semantic/image latent
  ↓
Channel ModNet
  ↓
Rate ModNet
  ↓
AWGN
  ↓
decoder
  ↓
Kodak reconstruction
```

The next experiment should **not** modify the Swin encoder yet.

The next layer we can insert is:

```text
SwinJSCC continuous latent
             ↓
       vector quantizer
             ↓
        codebook index
             ↓
      codebook reconstruction
             ↓
          channel
```

That will let us measure exactly what the codebook does to the continuous SwinJSCC representation.


# 16. Vector-Quantized SwinJSCC — Codebook Experiment

We now add a learned vector-quantization stage **after the trained SwinJSCC encoder**.

For a 256×256 image, the continuous SwinJSCC latent is:

`[B, 256, 320]`

That means 256 spatial latent vectors, each with 320 floating-point features.

A codebook contains `K` learned prototype vectors:

`[K, 320]`

For every latent vector `z[n]`, vector quantization finds the nearest prototype:

`k* = argmin_k ||z[n] - e[k]||²`

and represents that vector by the integer index `k*`.

The receiver reconstructs the latent using:

`index → codebook vector`

The codebook is shared model state. It is therefore reported separately from the per-image index payload.


In [ ]:
# ============================================================
# CODEBOOK CONFIGURATION
# ============================================================

CODEBOOK_DIM = 320
CODEBOOK_SIZE = 256

CODEBOOK_TRAIN_IMAGES = min(1000, len(train_files))
CODEBOOK_BATCH_SIZE = 16
CODEBOOK_EPOCHS = 5
CODEBOOK_LR = 1e-4
CODEBOOK_BETA = 0.25

CODEBOOK_SNR = 10
CODEBOOK_C = 96

CODEBOOK_CHECKPOINT_DIR = Path(
    "/kaggle/working/swinjscc_codebooks"
)
CODEBOOK_CHECKPOINT_DIR.mkdir(
    parents=True,
    exist_ok=True
)

print("K =", CODEBOOK_SIZE)
print("D =", CODEBOOK_DIM)
print("Training images =", CODEBOOK_TRAIN_IMAGES)


## 17. Trainable vector quantizer

We use the standard VQ objective with a straight-through estimator.

For the selected codebook vector `e`:

`L = ||sg(z) - e||² + beta ||z - sg(e)||²`

The first term updates the codebook.

The second term encourages the encoder representation to commit to the selected codebook vector.

For this first experiment we freeze SwinJSCC and train the codebook independently. This makes the quantization penalty measurable before we consider joint fine-tuning.


In [ ]:
class VectorQuantizer(nn.Module):

    def __init__(
        self,
        num_embeddings,
        embedding_dim,
        beta=0.25
    ):
        super().__init__()

        self.num_embeddings = num_embeddings
        self.embedding_dim = embedding_dim
        self.beta = beta

        self.embedding = nn.Embedding(
            num_embeddings,
            embedding_dim
        )

        nn.init.normal_(
            self.embedding.weight,
            mean=0.0,
            std=0.02
        )

    def encode_indices(self, z):
        # z: [B, N, D]
        if z.shape[-1] != self.embedding_dim:
            raise ValueError(
                f"Expected D={self.embedding_dim}, "
                f"got D={z.shape[-1]}"
            )

        flat = z.reshape(-1, self.embedding_dim)

        x2 = flat.pow(2).sum(
            dim=1,
            keepdim=True
        )

        e2 = self.embedding.weight.pow(2).sum(
            dim=1
        ).unsqueeze(0)

        distances = (
            x2
            + e2
            - 2.0 * flat @ self.embedding.weight.t()
        )

        indices = distances.argmin(dim=1)

        return indices.reshape(
            z.shape[0],
            z.shape[1]
        )

    def quantize(self, z, indices=None):

        if indices is None:
            indices = self.encode_indices(z)

        z_q = self.embedding(indices)

        return z_q, indices

    def forward(self, z):

        z_q, indices = self.quantize(z)

        codebook_loss = F.mse_loss(
            z_q,
            z.detach()
        )

        commitment_loss = F.mse_loss(
            z,
            z_q.detach()
        )

        loss = (
            codebook_loss
            + self.beta * commitment_loss
        )

        # Straight-through estimator.
        z_st = z + (z_q - z).detach()

        return z_st, indices, loss


In [ ]:
# ============================================================
# CODEBOOK SMOKE TEST
# ============================================================

vq_test = VectorQuantizer(
    CODEBOOK_SIZE,
    CODEBOOK_DIM,
    CODEBOOK_BETA
).cuda(0)

test_z = torch.randn(
    2,
    256,
    CODEBOOK_DIM,
    device="cuda:0"
)

test_zq, test_indices, test_loss = vq_test(test_z)

print("z       :", tuple(test_z.shape))
print("z_q     :", tuple(test_zq.shape))
print("indices :", tuple(test_indices.shape))
print("loss    :", float(test_loss))

assert test_zq.shape == test_z.shape
assert test_indices.shape == (2, 256)
assert int(test_indices.min()) >= 0
assert int(test_indices.max()) < CODEBOOK_SIZE

print("CODEBOOK SMOKE TEST PASSED")


## 18. Collect real SwinJSCC latents

The codebook should represent the distribution produced by our actual trained SwinJSCC encoder.

Therefore we freeze SwinJSCC and extract its continuous latents from DIV2K crops.

Only the codebook is trained in the next cell.


In [ ]:
class CodebookLatentDataset(Dataset):

    def __init__(
        self,
        files,
        model,
        max_images,
        snr,
        rate
    ):
        self.files = list(files)[:max_images]
        self.model = model
        self.snr = snr
        self.rate = rate
        self.latents = []

        self.extract()

    @torch.no_grad()
    def extract(self):

        self.model.eval()

        for i, path in enumerate(self.files):

            with Image.open(path) as im:
                im = im.convert("RGB")

                w, h = im.size

                x0 = random.randint(
                    0,
                    w - IMAGE_SIZE
                )
                y0 = random.randint(
                    0,
                    h - IMAGE_SIZE
                )

                crop = im.crop((
                    x0,
                    y0,
                    x0 + IMAGE_SIZE,
                    y0 + IMAGE_SIZE
                ))

                tensor = transforms.ToTensor()(
                    crop
                ).unsqueeze(0).cuda(0)

            _, feature, _ = self.model(
                tensor,
                self.snr,
                self.rate
            )

            self.latents.append(
                feature[0].detach().cpu()
            )

            if (i + 1) % 100 == 0 or i + 1 == len(self.files):
                print(
                    f"Extracted {i+1}/{len(self.files)}"
                )

    def __len__(self):
        return len(self.latents)

    def __getitem__(self, index):
        return self.latents[index]


codebook_latent_dataset = CodebookLatentDataset(
    train_files,
    sara_model,
    CODEBOOK_TRAIN_IMAGES,
    CODEBOOK_SNR,
    CODEBOOK_C
)

codebook_loader = DataLoader(
    codebook_latent_dataset,
    batch_size=CODEBOOK_BATCH_SIZE,
    shuffle=True,
    num_workers=2,
    pin_memory=True
)

print("Latent samples:", len(codebook_latent_dataset))
print("Latent shape:", tuple(codebook_latent_dataset[0].shape))


In [ ]:
# ============================================================
# CODEBOOK TRAINING
# ============================================================

vq_model = VectorQuantizer(
    CODEBOOK_SIZE,
    CODEBOOK_DIM,
    CODEBOOK_BETA
).cuda(0)

vq_optimizer = torch.optim.Adam(
    vq_model.parameters(),
    lr=CODEBOOK_LR
)

vq_history = []

for epoch in range(CODEBOOK_EPOCHS):

    vq_model.train()

    total_loss = 0.0
    total_samples = 0
    start = time.time()

    for z_cpu in codebook_loader:

        z = z_cpu.cuda(
            0,
            non_blocking=True
        )

        vq_optimizer.zero_grad(
            set_to_none=True
        )

        _, _, loss = vq_model(z)

        loss.backward()
        vq_optimizer.step()

        total_loss += (
            float(loss.detach())
            * z.shape[0]
        )

        total_samples += z.shape[0]

    epoch_loss = (
        total_loss
        /
        max(total_samples, 1)
    )

    vq_history.append(epoch_loss)

    print(
        f"[VQ] Epoch {epoch+1}/{CODEBOOK_EPOCHS} | "
        f"Loss={epoch_loss:.6f} | "
        f"{time.time()-start:.1f}s"
    )


In [ ]:
plt.figure(figsize=(8, 5))
plt.plot(
    range(1, len(vq_history) + 1),
    vq_history,
    marker="o"
)
plt.xlabel("Epoch")
plt.ylabel("VQ loss")
plt.title(f"Codebook Training Loss — K={CODEBOOK_SIZE}")
plt.grid(True)
plt.show()


In [ ]:
# ============================================================
# CODEBOOK UTILIZATION
# ============================================================

vq_model.eval()

usage = torch.zeros(
    CODEBOOK_SIZE,
    dtype=torch.long
)

with torch.no_grad():

    for z_cpu in codebook_loader:

        z = z_cpu.cuda(
            0,
            non_blocking=True
        )

        indices = (
            vq_model.encode_indices(z)
            .flatten()
            .cpu()
        )

        usage += torch.bincount(
            indices,
            minlength=CODEBOOK_SIZE
        )

used_entries = int((usage > 0).sum())

print(
    f"Used entries: {used_entries}/{CODEBOOK_SIZE}"
)
print(
    f"Utilization: {100*used_entries/CODEBOOK_SIZE:.2f}%"
)

plt.figure(figsize=(10, 4))
plt.bar(
    np.arange(CODEBOOK_SIZE),
    usage.numpy()
)
plt.xlabel("Codebook entry")
plt.ylabel("Assignments")
plt.title("Codebook Usage")
plt.show()


In [ ]:
vq_checkpoint = {
    "state_dict": vq_model.state_dict(),
    "codebook_size": CODEBOOK_SIZE,
    "codebook_dim": CODEBOOK_DIM,
    "beta": CODEBOOK_BETA,
    "source_model": MODEL_FULL,
    "latent_tokens": 256,
    "snr": CODEBOOK_SNR,
    "C": CODEBOOK_C,
}

vq_path = (
    CODEBOOK_CHECKPOINT_DIR
    / f"codebook_K{CODEBOOK_SIZE}.pt"
)

torch.save(vq_checkpoint, vq_path)

print("Saved:", vq_path)


# 19. Continuous vs codebook reconstruction

We now compare two paths using the same image, SNR and C:

```text
CONTINUOUS:
image → SwinJSCC encoder → z → decoder

CODEBOOK:
image → SwinJSCC encoder → z
      → nearest codebook vector
      → decoder
```

The second path introduces only the codebook quantization error.


In [ ]:
@torch.no_grad()
def evaluate_continuous_vs_codebook(
    model,
    vq,
    loader,
    snr,
    rate
):

    model.eval()
    vq.eval()

    core = unwrap(model)
    rows = []

    for images, names in loader:

        images = images.cuda(
            0,
            non_blocking=True
        )

        continuous_recon, z, mask = model(
            images,
            snr,
            rate
        )

        z_q, indices = vq.quantize(z)

        # Preserve the same active-channel mask.
        z_q = z_q * mask

        quantized_recon = core.decoder(
            z_q,
            snr,
            MODEL_FULL
        ).clamp(0.0, 1.0)

        mse_cont = F.mse_loss(
            continuous_recon,
            images
        ).item()

        mse_quant = F.mse_loss(
            quantized_recon,
            images
        ).item()

        psnr_cont = 10 * math.log10(
            1.0 / max(mse_cont, 1e-12)
        )

        psnr_quant = 10 * math.log10(
            1.0 / max(mse_quant, 1e-12)
        )

        rows.append({
            "image": names[0],
            "SNR_dB": snr,
            "C": rate,
            "continuous_MSE": mse_cont,
            "continuous_PSNR_dB": psnr_cont,
            "codebook_MSE": mse_quant,
            "codebook_PSNR_dB": psnr_quant,
            "PSNR_drop_dB": psnr_cont - psnr_quant,
        })

    return pd.DataFrame(rows)


codebook_eval_df = evaluate_continuous_vs_codebook(
    sara_model,
    vq_model,
    test_loader,
    CODEBOOK_SNR,
    CODEBOOK_C
)

display(codebook_eval_df)

print(
    "Mean continuous PSNR:",
    codebook_eval_df["continuous_PSNR_dB"].mean()
)
print(
    "Mean codebook PSNR:",
    codebook_eval_df["codebook_PSNR_dB"].mean()
)
print(
    "Mean PSNR drop:",
    codebook_eval_df["PSNR_drop_dB"].mean()
)


# 20. Exact representation-size comparison

For a 256×256 RGB image:

\[
256\times256\times3\times8
=
1,572,864\text{ bits}
\]

= 192 KiB.

The continuous latent contains:

\[
256\times320=81,920
\]

FP32 values, so its PyTorch FP32 memory representation is:

\[
81,920\times32
=
2,621,440\text{ bits}
\]

= 320 KiB.

This is **latent memory**, not automatically a transmitted payload.

For a codebook with K entries:

\[
b_{index}=\lceil\log_2K\rceil.
\]

There are 256 indices per image:

\[
B_{indices}=256\,b_{index}.
\]

The shared codebook contains:

\[
K\times320
\]

FP32 values.

Its one-time cost is therefore:

\[
B_{CB}=K\times320\times32.
\]

If the same codebook is reused for M images, the amortized codebook cost is:

\[
B_{CB}/M.
\]

The table below reports all of these quantities separately.


In [ ]:
def latent_size_report(
    image_size=256,
    latent_tokens=256,
    latent_dim=320,
    C_values=(32, 64, 96, 128, 192),
    K_values=(64, 256, 1024),
    amortization_images=None
):

    if amortization_images is None:
        amortization_images = max(
            len(train_files),
            1
        )

    original_bits = (
        image_size * image_size * 3 * 8
    )

    rows = [{
        "Representation": "Original RGB image",
        "Setting": "-",
        "Bits_per_image": original_bits,
        "Bytes_per_image": original_bits / 8,
        "KiB_per_image": original_bits / 8 / 1024,
        "bpp": original_bits / (image_size * image_size),
        "Compression_ratio_vs_original": 1.0,
        "Shared_codebook_KiB": 0.0,
        "Amortized_total_KiB": original_bits / 8 / 1024,
    }]

    continuous_values = latent_tokens * latent_dim
    continuous_bits = continuous_values * 32

    rows.append({
        "Representation": "Continuous latent FP32",
        "Setting": f"{latent_tokens}×{latent_dim}",
        "Bits_per_image": continuous_bits,
        "Bytes_per_image": continuous_bits / 8,
        "KiB_per_image": continuous_bits / 8 / 1024,
        "bpp": continuous_bits / (image_size * image_size),
        "Compression_ratio_vs_original": (
            original_bits / continuous_bits
        ),
        "Shared_codebook_KiB": 0.0,
        "Amortized_total_KiB": continuous_bits / 8 / 1024,
    })

    for C in C_values:

        active_values = latent_tokens * C
        active_bits = active_values * 32

        rows.append({
            "Representation": "Active latent FP32",
            "Setting": f"C={C}",
            "Bits_per_image": active_bits,
            "Bytes_per_image": active_bits / 8,
            "KiB_per_image": active_bits / 8 / 1024,
            "bpp": active_bits / (image_size * image_size),
            "Compression_ratio_vs_original": (
                original_bits / active_bits
            ),
            "Shared_codebook_KiB": 0.0,
            "Amortized_total_KiB": active_bits / 8 / 1024,
        })

    for K in K_values:

        bits_per_index = math.ceil(
            math.log2(K)
        )

        payload_bits = (
            latent_tokens * bits_per_index
        )

        codebook_bits = (
            K * latent_dim * 32
        )

        amortized_bits = (
            payload_bits
            + codebook_bits / amortization_images
        )

        rows.append({
            "Representation": "Codebook indices",
            "Setting": f"K={K}",
            "Bits_per_image": payload_bits,
            "Bytes_per_image": payload_bits / 8,
            "KiB_per_image": payload_bits / 8 / 1024,
            "bpp": payload_bits / (image_size * image_size),
            "Compression_ratio_vs_original": (
                original_bits / payload_bits
            ),
            "Shared_codebook_KiB": (
                codebook_bits / 8 / 1024
            ),
            "Amortized_total_KiB": (
                amortized_bits / 8 / 1024
            ),
        })

    return pd.DataFrame(rows)


size_df = latent_size_report(
    amortization_images=max(
        len(train_files),
        1
    )
)

display(size_df)


In [ ]:
# ============================================================
# SIZE / COMPRESSION VISUALIZATION
# ============================================================

plt.figure(figsize=(12, 6))

labels = [
    f"{r['Representation']}\n{r['Setting']}"
    for _, r in size_df.iterrows()
]

plt.bar(
    np.arange(len(size_df)),
    size_df["KiB_per_image"]
)

plt.xticks(
    np.arange(len(size_df)),
    labels,
    rotation=55,
    ha="right"
)

plt.ylabel("KiB per image")
plt.title("Continuous / Active / Codebook Representation Size")
plt.grid(axis="y")
plt.tight_layout()
plt.show()


codebook_rows = size_df[
    size_df["Representation"] == "Codebook indices"
]

plt.figure(figsize=(8, 5))

plt.bar(
    codebook_rows["Setting"],
    codebook_rows["bpp"]
)

plt.xlabel("Codebook size K")
plt.ylabel("Index payload bpp")
plt.title("Codebook Index Payload")
plt.grid(axis="y")
plt.show()


In [ ]:
# ============================================================
# CODEBOOK SUMMARY TABLE
# ============================================================

summary = []

for K in (64, 256, 1024):

    row = size_df[
        (size_df["Representation"] == "Codebook indices")
        & (size_df["Setting"] == f"K={K}")
    ].iloc[0]

    summary.append({
        "K": K,
        "bits_per_index": math.ceil(math.log2(K)),
        "indices_per_image": 256,
        "index_payload_bytes": row["Bytes_per_image"],
        "index_payload_KiB": row["KiB_per_image"],
        "index_payload_bpp": row["bpp"],
        "payload_compression_ratio": (
            row["Compression_ratio_vs_original"]
        ),
        "shared_codebook_KiB": (
            row["Shared_codebook_KiB"]
        ),
        "amortized_total_KiB": (
            row["Amortized_total_KiB"]
        ),
    })

codebook_summary_df = pd.DataFrame(summary)

display(codebook_summary_df)


In [ ]:
# ============================================================
# REPRESENTATIVE ORIGINAL / CONTINUOUS / CODEBOOK
# ============================================================

@torch.no_grad()
def show_codebook_reconstruction(
    model,
    vq,
    dataset,
    index=0,
    snr=10,
    rate=96
):

    model.eval()
    vq.eval()

    original, name = dataset[index]

    image = original.unsqueeze(
        0
    ).cuda(0)

    continuous_recon, z, mask = model(
        image,
        snr,
        rate
    )

    z_q, indices = vq.quantize(z)
    z_q = z_q * mask

    core = unwrap(model)

    quantized_recon = core.decoder(
        z_q,
        snr,
        MODEL_FULL
    ).clamp(0.0, 1.0)

    continuous_recon = continuous_recon[0].cpu()
    quantized_recon = quantized_recon[0].cpu()

    fig, axes = plt.subplots(
        1, 3,
        figsize=(15, 5)
    )

    axes[0].imshow(
        original.permute(1, 2, 0)
    )
    axes[0].set_title(
        f"Original\n{name}"
    )

    axes[1].imshow(
        continuous_recon.permute(1, 2, 0)
    )
    axes[1].set_title(
        "Continuous latent"
    )

    axes[2].imshow(
        quantized_recon.permute(1, 2, 0)
    )
    axes[2].set_title(
        f"Codebook latent\nK={CODEBOOK_SIZE}"
    )

    for ax in axes:
        ax.axis("off")

    plt.tight_layout()
    plt.show()


show_codebook_reconstruction(
    sara_model,
    vq_model,
    test_dataset,
    index=0,
    snr=CODEBOOK_SNR,
    rate=CODEBOOK_C
)


In [ ]:
# ============================================================
# SAVE CODEBOOK RESULTS
# ============================================================

codebook_eval_df.to_csv(
    CODEBOOK_CHECKPOINT_DIR
    / f"kodak_codebook_K{CODEBOOK_SIZE}.csv",
    index=False
)

size_df.to_csv(
    CODEBOOK_CHECKPOINT_DIR
    / "complete_latent_size_comparison.csv",
    index=False
)

codebook_summary_df.to_csv(
    CODEBOOK_CHECKPOINT_DIR
    / "codebook_payload_summary.csv",
    index=False
)

print("Saved codebook experiment artifacts:")
for p in sorted(CODEBOOK_CHECKPOINT_DIR.iterdir()):
    print(" ", p)


# 23. What this experiment establishes

We now have a clean baseline comparison:

```text
                 SwinJSCC encoder
                        │
                        ▼
                continuous z
                  [256,320]
                    /   \
                   /     \
                  ▼       ▼
           continuous    VQ
               │          │
               │       index[256]
               │          │
               │       codebook
               │          │
               │          ▼
               │       quantized z
               │          │
               └────┬─────┘
                    ▼
                 decoder
                    ▼
              reconstruction
```

The key quantities are:

1. Continuous latent FP32 memory.
2. Active latent FP32 memory for each `C`.
3. Codebook index payload for each `K`.
4. One-time shared codebook size.
5. Amortized codebook overhead.
6. Reconstruction PSNR penalty caused by quantization.

For the later semantic scheduler, the **codebook index sequence** is particularly useful because each image is no longer represented only by a dense floating-point tensor. It becomes a sequence of discrete latent symbols that can be assigned priorities and scheduled individually.

Do not yet claim that codebook indices are "semantic importance" by themselves. The next research step is to determine what useful context/novelty/relevance information can be extracted from the SwinJSCC latent and/or codebook assignments.
